# Tool-Call Fine-Tune Lab — QLoRA Pipeline

Fine-tunes **Qwen2.5-7B-Instruct** on a curated tool-calling dataset using QLoRA (4-bit NF4) on Kaggle GPU runtimes.

**Dataset:** `doeonkim00/tool-call-finetune-data` (attached as Kaggle dataset input)

**Expected runtime:** ~8–10 hours on a compatible T4-class accelerator. Unsupported accelerators automatically fall back to a smoke-only proof path.

**Stack:** `transformers`, `peft`, `datasets`, `bitsandbytes`.

Internet should remain enabled so the notebook can download the base model when no cached Kaggle model input is attached. Unsupported GPUs now switch to a non-benchmark smoke mode instead of crashing the notebook run.


In [ ]:
# Cell 1 — Install / upgrade missing packages and inspect the runtime
import subprocess
import sys


def run(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode, result.stdout.strip(), result.stderr.strip()


def parse_compute_capability(value):
    if not value:
        return None
    try:
        major, minor = value.split(".", 1)
        return int(major), int(minor)
    except Exception:
        return None


def detect_gpu_runtime():
    code, stdout, _ = run([
        "nvidia-smi",
        "--query-gpu=name,compute_cap",
        "--format=csv,noheader",
    ])
    if code != 0 or not stdout:
        return None, None
    first_line = stdout.splitlines()[0]
    parts = [part.strip() for part in first_line.split(",")]
    name = parts[0] if parts else None
    capability = parts[1] if len(parts) > 1 else None
    return name, capability


GPU_NAME_RAW, GPU_COMPUTE_CAPABILITY_RAW = detect_gpu_runtime()
GPU_COMPUTE_CAPABILITY = parse_compute_capability(GPU_COMPUTE_CAPABILITY_RAW)
GPU_SUPPORTS_FULL_QLORA = GPU_COMPUTE_CAPABILITY is not None and GPU_COMPUTE_CAPABILITY >= (7, 0)

print(f"Detected GPU name: {GPU_NAME_RAW}")
print(f"Detected GPU compute capability: {GPU_COMPUTE_CAPABILITY_RAW}")
print(f"Full 4-bit QLoRA path available: {GPU_SUPPORTS_FULL_QLORA}")

packages = ["bitsandbytes>=0.46.1", "peft>=0.13.0", "accelerate>=1.0.0"]
for pkg in packages:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", pkg, "-q"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"  FAILED: {result.stderr.strip()[:200]}")
    else:
        print("  OK")

if GPU_SUPPORTS_FULL_QLORA:
    try:
        import bitsandbytes as bnb
        print(f"\nbitsandbytes version: {bnb.__version__}")
    except ImportError as exc:
        print(f"\nFATAL: bitsandbytes not available: {exc}")
        raise
else:
    print("\nSkipping bitsandbytes runtime validation because this accelerator will use smoke mode.")


In [ ]:
# Cell 2 — Imports + config

import json
import os
from pathlib import Path

import torch


def parse_compute_capability(value):
    if not value:
        return None
    try:
        major, minor = value.split(".", 1)
        return int(major), int(minor)
    except Exception:
        return None


GPU_NAME_RAW = globals().get("GPU_NAME_RAW")
GPU_COMPUTE_CAPABILITY_RAW = globals().get("GPU_COMPUTE_CAPABILITY_RAW")
GPU_COMPUTE_CAPABILITY = parse_compute_capability(GPU_COMPUTE_CAPABILITY_RAW)

SMOKE_ONLY = False
SMOKE_REASON = None

if not torch.cuda.is_available():
    SMOKE_ONLY = True
    SMOKE_REASON = "CUDA is unavailable in this Kaggle runtime."
elif GPU_COMPUTE_CAPABILITY is None:
    SMOKE_ONLY = True
    SMOKE_REASON = "Unable to determine GPU compute capability for this Kaggle runtime."
elif GPU_COMPUTE_CAPABILITY < (7, 0):
    SMOKE_ONLY = True
    SMOKE_REASON = (
        f"GPU {GPU_NAME_RAW} (compute capability {GPU_COMPUTE_CAPABILITY_RAW}) "
        "is below the supported threshold for this 4-bit QLoRA path."
    )

print(f"Smoke mode active: {SMOKE_ONLY}")
if SMOKE_REASON:
    print(f"Smoke reason: {SMOKE_REASON}")

# ---------------------------------------------------------------------------
# Kaggle Secrets (WANDB_API_KEY and HF_TOKEN are optional)
# ---------------------------------------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        WANDB_API_KEY = _secrets.get_secret("WANDB_API_KEY")
    except Exception:
        WANDB_API_KEY = None
    try:
        HF_TOKEN = _secrets.get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
except ImportError:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY")
    HF_TOKEN = os.environ.get("HF_TOKEN")

print(f"W&B key present: {bool(WANDB_API_KEY)}")
print(f"HF token present: {bool(HF_TOKEN)}")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = Path("/kaggle/input/tool-call-finetune-data")
OUTPUT_DIR = Path("/kaggle/working")
LORA_DIR = OUTPUT_DIR / "qwen25-7b-tool-call-lora"
MERGED_DIR = OUTPUT_DIR / "qwen25-7b-tool-call-merged"
RESULTS_DIR = OUTPUT_DIR / "results"

for directory in [LORA_DIR, MERGED_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Model path: scan /kaggle/input for config.json to find an attached cached model
# ---------------------------------------------------------------------------
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
HF_MODEL_ID = BASE_MODEL

kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    print(f"Scanning {kaggle_input} for Qwen model...")
    for config_path in sorted(kaggle_input.rglob("config.json")):
        try:
            config = json.loads(config_path.read_text())
            model_type = config.get("model_type", "").lower()
            name_or_path = config.get("_name_or_path", "")
            if "qwen2" in model_type or "Qwen" in name_or_path:
                BASE_MODEL = str(config_path.parent)
                print(f"Found cached model at {BASE_MODEL}")
                break
        except Exception:
            continue

print(f"Base model source: {BASE_MODEL}")

# ---------------------------------------------------------------------------
# Hyperparameters
# ---------------------------------------------------------------------------
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 2e-4
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
WARMUP_RATIO = 0.03

print("Training config:")
print(f"  epochs={EPOCHS}")
print(f"  batch_size={BATCH_SIZE}")
print(f"  grad_accum={GRAD_ACCUM}")
print(f"  learning_rate={LR}")
print(f"  max_seq_length={MAX_SEQ_LENGTH}")


In [ ]:
# Cell 3 — Load data from Kaggle dataset path


def load_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


train_data = load_jsonl(DATA_DIR / "train.jsonl")
val_data = load_jsonl(DATA_DIR / "val.jsonl")
test_data = load_jsonl(DATA_DIR / "test.jsonl")

print(f"Train: {len(train_data):,} examples")
print(f"Val:   {len(val_data):,} examples")
print(f"Test:  {len(test_data):,} examples")

sample = train_data[0] if train_data else {}
print(f"\nSample source:   {sample.get('source', 'n/a')}")
print(f"Sample category: {sample.get('category', 'n/a')}")
messages = sample.get("messages", [])
if len(messages) > 1:
    print(f"User msg:        {messages[1]['content'][:120]}")
if len(messages) > 2:
    print(f"Tool calls:      {len(messages[2].get('tool_calls', []))}")


In [ ]:
# Cell 4 — Configure training (or smoke-mode guard)

from transformers import BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig

if SMOKE_ONLY:
    bnb_config = None
    lora_config = None
    training_args = None
    print("Smoke mode active: skipping 4-bit training config because this Kaggle accelerator is not compatible with the full QLoRA path.")
else:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )

    training_args = TrainingArguments(
        output_dir=str(LORA_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_ratio=WARMUP_RATIO,
        fp16=True,
        bf16=False,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="wandb" if WANDB_API_KEY else "none",
        run_name="qwen25-7b-tool-call-lora-t4",
        remove_unused_columns=False,
    )

    print("BnB config:")
    print(f"  quant_type={bnb_config.bnb_4bit_quant_type}, compute_dtype=fp16, double_quant=True")
    print("LoRA config:")
    print(f"  r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
    print("Training args:")
    print(f"  epochs={EPOCHS}, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LR}")
    print("  fp16=True, gradient_checkpointing=True")


In [ ]:
# Cell 5 — Load tokenizer and optionally the 4-bit base model

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, prepare_model_for_kbit_training

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = None
if SMOKE_ONLY:
    print("Smoke mode active: skipping quantized model load and LoRA attachment.")
else:
    print("Loading model in 4-bit...")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    print(f"\nGPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


In [ ]:
# Cell 6 — Format dataset for SFT training (ChatML format with tool_call markup)

from datasets import Dataset


def format_for_training(example):
    """Convert a JSONL record to a single ChatML string for Trainer."""
    messages = example.get("messages", [])
    tools = example.get("tools", [])

    tool_desc = ""
    if tools:
        tool_desc = "\n\nAvailable tools:"
        for tool in tools:
            fn = tool.get("function", {})
            tool_desc += f"\n- {fn.get('name', '')}: {fn.get('description', '')}"

    parts = []
    for message in messages:
        role = message["role"]
        content = message.get("content") or ""

        if role == "system":
            parts.append(f"<|im_start|>system\n{content}{tool_desc}<|im_end|>")
        elif role == "user":
            parts.append(f"<|im_start|>user\n{content}<|im_end|>")
        elif role == "tool":
            parts.append(f"<|im_start|>tool\n{content}<|im_end|>")
        elif role == "assistant":
            tool_calls = message.get("tool_calls", [])
            if tool_calls:
                tool_call_blocks = []
                for tool_call in tool_calls:
                    fn = tool_call.get("function", {})
                    arguments = fn.get("arguments", {})
                    if isinstance(arguments, str):
                        try:
                            arguments = json.loads(arguments)
                        except json.JSONDecodeError:
                            pass
                    tool_call_blocks.append(json.dumps({"name": fn.get("name", ""), "arguments": arguments}))
                assistant_content = "<tool_call>\n" + "\n".join(tool_call_blocks) + "\n</tool_call>"
            else:
                assistant_content = content
            parts.append(f"<|im_start|>assistant\n{assistant_content}<|im_end|>")

    return {"text": "\n".join(parts)}


train_formatted = [format_for_training(example) for example in train_data]
val_formatted = [format_for_training(example) for example in val_data]

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print(f"Formatted train rows: {len(train_dataset):,}")
print(f"Formatted val rows: {len(val_dataset):,}")
print("Formatted train sample preview:")
print(train_dataset[0]["text"][:600])


In [ ]:
# Cell 7 — Train with transformers.Trainer (or run smoke-mode preprocessing)

from transformers import DataCollatorForLanguageModeling, Trainer

if WANDB_API_KEY and not SMOKE_ONLY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    os.environ["WANDB_PROJECT"] = "tool-call-finetune-lab"
    print("W&B logged in")
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled")


def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )


print("Tokenizing datasets...")
train_tokenized = train_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train",
)
val_tokenized = val_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing val",
)


def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples


train_tokenized = train_tokenized.map(add_labels, desc="Adding labels train")
val_tokenized = val_tokenized.map(add_labels, desc="Adding labels val")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = None
if SMOKE_ONLY:
    print("Smoke mode active: skipping Trainer.fit while preserving tokenization proof.")
    print(f"Tokenized train rows: {len(train_tokenized):,}")
    print(f"Tokenized val rows: {len(val_tokenized):,}")
else:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        data_collator=data_collator,
    )

    steps_per_epoch = max(1, len(train_tokenized) // max(1, BATCH_SIZE * GRAD_ACCUM))
    print(f"Training {len(train_tokenized):,} examples for {EPOCHS} epoch(s)")
    print(f"Steps per epoch: {steps_per_epoch}")
    print(f"GPU memory before train: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

    trainer.train()
    trainer.save_model(str(LORA_DIR))
    tokenizer.save_pretrained(str(LORA_DIR))

    print(f"Training complete. LoRA adapter saved to {LORA_DIR}")


In [ ]:
# Cell 8 — Merge LoRA adapter into base model (full path only)

if SMOKE_ONLY:
    print("Smoke mode active: skipping LoRA merge step.")
else:
    from peft import PeftModel

    print("Loading base model for merging (on CPU to preserve VRAM)...")
    base_model_for_merge = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True,
    )

    print("Attaching LoRA adapter...")
    peft_model = PeftModel.from_pretrained(base_model_for_merge, str(LORA_DIR))

    print("Merging weights...")
    merged_model = peft_model.merge_and_unload()

    merged_model.save_pretrained(str(MERGED_DIR))
    tokenizer.save_pretrained(str(MERGED_DIR))

    print(f"Merged model saved to {MERGED_DIR}")

    del base_model_for_merge, peft_model
    torch.cuda.empty_cache()


In [ ]:
# Cell 9 — Quick eval: test a few tool-call prompts (full path only)

quick_eval_outputs = []

if SMOKE_ONLY:
    print("Smoke mode active: skipping merged-model inference smoke test.")
else:
    from transformers import pipeline as hf_pipeline

    print("Loading merged model for inference...")
    pipe = hf_pipeline(
        "text-generation",
        model=str(MERGED_DIR),
        tokenizer=tokenizer,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    quick_prompts = [
        (
            "get_weather",
            "<|im_start|>system\nYou are a helpful assistant.\n\nAvailable tools:\n- get_weather: Get current weather for a location<|im_end|>\n"
            "<|im_start|>user\nWhat's the weather in Tokyo?<|im_end|>\n<|im_start|>assistant\n",
        ),
        (
            "search_web",
            "<|im_start|>system\nYou are a helpful assistant.\n\nAvailable tools:\n- search_web: Search the internet<|im_end|>\n"
            "<|im_start|>user\nFind the latest news about AI.<|im_end|>\n<|im_start|>assistant\n",
        ),
    ]

    for expected_fn, prompt in quick_prompts:
        result = pipe(prompt, max_new_tokens=200, do_sample=False)
        generated = result[0]["generated_text"][len(prompt):]
        hit = expected_fn in generated
        quick_eval_outputs.append(
            {
                "expected_function": expected_fn,
                "matched": hit,
                "generated_excerpt": generated[:300],
            }
        )
        print(f"\n[{'PASS' if hit else 'FAIL'}] Expected function: {expected_fn}")
        print(f"Output: {generated[:300]}")


In [ ]:
# Cell 10 — BFCL eval (full path) or smoke summary (fallback path)

bfcl_test = [example for example in test_data if example.get("source") == "bfcl"]
print(f"BFCL test examples total: {len(bfcl_test)}")

if SMOKE_ONLY:
    correct = 0
    total = 0
    overall_acc = 0.0
    results_by_category = {}
    smoke_summary = {
        "mode": "smoke_only",
        "reason": SMOKE_REASON,
        "gpu_name": GPU_NAME_RAW,
        "gpu_compute_capability": GPU_COMPUTE_CAPABILITY_RAW,
        "train_examples": len(train_data),
        "val_examples": len(val_data),
        "test_examples": len(test_data),
        "bfcl_examples_available": len(bfcl_test),
        "base_model": BASE_MODEL,
        "note": "This Kaggle runtime validated data loading, tokenizer setup, and preprocessing without attempting unsupported 4-bit QLoRA training.",
    }
    print(json.dumps(smoke_summary, indent=2))
else:
    smoke_summary = None
    correct = 0
    total = 0
    results_by_category = {}

    for example in bfcl_test[:100]:
        category = example.get("category", "unknown")
        results_by_category.setdefault(category, {"correct": 0, "total": 0})

        formatted = format_for_training(example)
        prompt = formatted["text"].rsplit("<|im_start|>assistant", 1)[0] + "<|im_start|>assistant\n"

        output = pipe(prompt, max_new_tokens=300, do_sample=False)
        generated = output[0]["generated_text"][len(prompt):]

        expected_calls = example["messages"][2].get("tool_calls", []) if len(example["messages"]) > 2 else []
        expected_names = {tool_call["function"]["name"] for tool_call in expected_calls}
        generated_names = {name for name in expected_names if name in generated}
        is_correct = generated_names == expected_names

        if is_correct:
            correct += 1
            results_by_category[category]["correct"] += 1
        total += 1
        results_by_category[category]["total"] += 1

    overall_acc = correct / total * 100 if total > 0 else 0.0

    print(f"\n{'=' * 60}")
    print("BFCL Eval Results (first 100 examples)")
    print(f"{'=' * 60}")
    print(f"Overall: {correct}/{total}  ({overall_acc:.1f}%)")
    print("\nPer category:")
    for category, stats in sorted(results_by_category.items()):
        accuracy = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
        print(f"  {category:30s}  {stats['correct']}/{stats['total']}  ({accuracy:.1f}%)")


In [ ]:
# Cell 11 — Save results + optionally push to HuggingFace

results_path = RESULTS_DIR / "bfcl_results.json"

if SMOKE_ONLY:
    eval_results = {
        "mode": "smoke_only",
        "reason": SMOKE_REASON,
        "data_snapshot": {
            "train_examples": len(train_data),
            "val_examples": len(val_data),
            "test_examples": len(test_data),
            "bfcl_examples_available": len(bfcl_test),
        },
        "runtime": {
            "gpu": GPU_NAME_RAW,
            "gpu_compute_capability": GPU_COMPUTE_CAPABILITY_RAW,
            "cuda_available": bool(torch.cuda.is_available()),
        },
        "note": "Unsupported Kaggle accelerators now fall back to a smoke-only proof path instead of crashing the public notebook run.",
    }
else:
    eval_results = {
        "model": f"{BASE_MODEL} + QLoRA",
        "overall_accuracy": f"{overall_acc:.1f}%",
        "correct": correct,
        "total": total,
        "categories": {
            key: {
                "correct": value["correct"],
                "total": value["total"],
                "accuracy": f"{value['correct'] / value['total'] * 100:.1f}%" if value["total"] > 0 else "n/a",
            }
            for key, value in results_by_category.items()
        },
        "training": {
            "epochs": EPOCHS,
            "train_examples": len(train_dataset),
            "val_examples": len(val_dataset),
            "gpu": GPU_NAME_RAW or "Kaggle GPU runtime",
            "fp16": True,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
        },
        "quick_eval": quick_eval_outputs,
    }

results_path.write_text(json.dumps(eval_results, indent=2))
print(f"Results saved to {results_path}")

if HF_TOKEN and not SMOKE_ONLY:
    from huggingface_hub import HfApi

    api = HfApi(token=HF_TOKEN)
    hf_repo_owner = os.environ.get("HF_REPO_OWNER", "doeonkim00")
    repo_id = f"{hf_repo_owner}/qwen2.5-7b-tool-calling-lora"
    api.create_repo(repo_id, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=str(LORA_DIR),
        repo_id=repo_id,
        commit_message=f"QLoRA adapter — BFCL acc {overall_acc:.1f}%",
    )
    print(f"LoRA adapter pushed to https://huggingface.co/{repo_id}")
elif SMOKE_ONLY:
    print("Smoke mode active — skipping Hugging Face push because no trained adapter was produced.")
else:
    print("HF_TOKEN not set — skipping Hugging Face push.")
    print("Add HF_TOKEN to Kaggle Secrets to enable automatic upload on full-path runs.")


## Summary

| Step | Detail |
|------|--------|
| Base model | Qwen2.5-7B-Instruct |
| Quantisation | 4-bit NF4 (QLoRA) on compatible accelerators |
| LoRA rank | 16, alpha 32 |
| Training | 1 epoch, fp16, batch 1 × grad_accum 8 |
| Hardware | Kaggle GPU runtime with smoke fallback on unsupported accelerators |
| Dataset | `doeonkim00/tool-call-finetune-data` |
| Eval | BFCL first-100 loose match on full path, smoke summary on fallback path |

**Next steps**
- Re-run on a T4-class accelerator for the full training + eval path
- Publish Hugging Face artifacts once the final model directory is available again
- Keep the public Kaggle notebook green even on unsupported accelerators by preserving the smoke fallback path
